# Chạy AI service (model v3) trên Colab GPU

Chạy **đúng code của branch `hocnv/DO-06`** với **model v3** (feature 007/008; đổi `MODEL` ở cell [1] để chạy v2/v1) trên GPU T4, rồi mở Cloudflare Tunnel để backend Spring Boot ở máy local gọi lên. Không cần tải model 1,1 GB về máy.

1. Runtime → Change runtime type → **T4 GPU** → Save.
2. Runtime → **Run all**. Cell [4] in URL `https://….trycloudflare.com`.
3. Chạy backend local trỏ vào URL đó:
   ```bash
   cd backend && ./mvnw spring-boot:run -Dspring-boot.run.arguments=--ai.service.url=https://….trycloudflare.com
   ```
4. Giữ tab Colab mở trong lúc test. Cell [6] theo dõi số request; dừng cell đó khi xong.

Ghi chú: Colab không có PostgreSQL nên KB được nạp từ seed CSV trong repo (cùng dữ liệu). Log của service chỉ ghi số đếm, không ghi nội dung CV.

In [ ]:
# [1] Cấu hình
MODEL = "v3"     # "v3" (mặc định), "v2" hoặc "v1" (model cũ, để so sánh)
REPO, BRANCH = "https://github.com/long20102004/recruit_system.git", "hocnv/DO-06"
# File GGUF trên Drive dự án (thư mục public cv-analysis-service)
GGUF_DRIVE_ID = {
    "v3": "1mqmolAF-4nJXg1db89oHAYZKYlZ3CUcq",   # finetune_output_v3/gguf_gguf/Qwen3-1.7B.Q4_K_M.gguf
    "v2": "1X4A2sVFDWbtIIP6hwjjJuiA3Sb-mjrUY",   # finetune_output_v2/gguf_gguf/Qwen3-1.7B.Q4_K_M.gguf
    "v1": "1Kt4BbL22tR1HLw16f2tyGyus6mQDI2SV",   # finetune_output/gguf_gguf/Qwen3-1.7B.Q4_K_M.gguf
}
GGUF_FILE = {"v3": "merged_model/model-v3-q4_k_m.gguf", "v2": "merged_model/model-v2-q4_k_m.gguf", "v1": "merged_model/model.gguf"}

In [ ]:
# [2] Code đúng branch + thư viện (llama-cpp-python CUDA 12, cloudflared)
%cd /content
!rm -rf /content/recruit_system
!git clone -q --depth 1 -b {BRANCH} {REPO} /content/recruit_system
%cd /content/recruit_system/ai-service
!git log --oneline -1
!pip install -q fastapi uvicorn pydantic json-repair rapidfuzz gdown
!pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Colab có thể không còn libcudart.so.12: đăng ký lib của các gói nvidia-* (pip) với dynamic linker
import glob, site, os
lib_dirs = sorted({d for sp in site.getsitepackages() for d in glob.glob(os.path.join(sp, "nvidia", "*", "lib"))})
open("/etc/ld.so.conf.d/nvidia-pip.conf", "w").write("\n".join(lib_dirs + ["/usr/local/cuda/lib64"]) + "\n")
!ldconfig
os.environ["LD_LIBRARY_PATH"] = ":".join(lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")])
!python -c "import llama_cpp; print('llama_cpp', llama_cpp.__version__, '| GPU offload:', llama_cpp.llama_supports_gpu_offload())"

In [ ]:
# [3] Tải model từ Drive dự án (~1,1 GB, khoảng 1 phút)
%cd /content/recruit_system/ai-service
import os
os.makedirs("merged_model", exist_ok=True)
path = GGUF_FILE[MODEL]
if not os.path.exists(path):
    import subprocess
    subprocess.run(["gdown", "-q", GGUF_DRIVE_ID[MODEL], "-O", path], check=True)
size = os.path.getsize(path)
print(path, size, "bytes")
assert size > 1_000_000_000 and open(path, "rb").read(4) == b"GGUF", "Tải model lỗi: xoá file và chạy lại cell"

In [ ]:
# [4] Chạy service + mở tunnel, in URL cho backend
%cd /content/recruit_system/ai-service
import subprocess, time, re, urllib.request, json, os
env = dict(os.environ, PROMPT_VERSION=MODEL, GGUF_MODEL_PATH=GGUF_FILE[MODEL])
env.pop("KB_DSN", None)   # Colab không có Postgres: dùng seed CSV
srv = subprocess.Popen(["python", "-m", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"],
                       stdout=open("server.log", "w"), stderr=subprocess.STDOUT, env=env)
h = None
for _ in range(120):
    try:
        h = json.load(urllib.request.urlopen("http://127.0.0.1:8000/health")); break
    except Exception:
        time.sleep(2)
print("health:", h)
assert h and h["engine"] == "llama_cpp" and h["prompt_version"] == MODEL, "Service chưa chạy đúng, xem server.log"

tun = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
                       stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        url = m.group(0); break
print("=" * 70)
print("URL ai-service:", url)
print("Backend local:  ./mvnw spring-boot:run -Dspring-boot.run.arguments=--ai.service.url=" + str(url))
print("=" * 70)

In [ ]:
# [5] Kiểm tra nhanh bằng một CV tổng hợp (không phải dữ liệu thật)
import time, json, urllib.request
cv = """NGUYEN VAN SYNTHETIC
syn.dev@example.test | 0900000099 | https://github.com/synthetic-dev
EXPERIENCE
Công ty TNHH Synthetic Software      October 2025 - May 2026
Backend Developer
• Developed RESTful APIs using Spring Boot and PostgreSQL
EDUCATION
Học viện Công nghệ Bưu chính Viễn thông      2022 - 2026
TECHNICAL SKILL
● Backend: Spring Boot, ReactJS, Postgress, Docker
LANGUAGES
● English: TOEIC 750
"""
t0 = time.time()
req = urllib.request.Request("http://127.0.0.1:8000/api/v1/extract-cv", data=json.dumps({"cv_text": cv}).encode(),
                             headers={"Content-Type": "application/json"})
d = json.load(urllib.request.urlopen(req, timeout=600))["data"]
print(f"{time.time() - t0:.1f} s")
print("experience:", [(e["company"], e["job_title"], e["start_date"]) for e in d["experience"]])
print("skills_normalized:", [(s["raw"], s["name"]) for s in d.get("skills_normalized", [])])
print("implied_skills:", [s["name"] for s in d.get("implied_skills", [])])
print("skills_detected:", [(s["name"], s["source"]) for s in d.get("skills_detected", [])])
print("languages:", d["languages"], "| flags:", d.get("flags"))

In [ ]:
# [6] Giữ session sống + theo dõi (dừng cell này khi test xong)
import time, re
while True:
    log = open("server.log").read()
    times = [float(x) for x in re.findall(r"Inference time: ([0-9.]+)s", log)]
    avg = f"{sum(times) / len(times):.1f}s" if times else "-"
    print(time.strftime("%H:%M:%S"), "| requests:", len(times), "| trung bình:", avg,
          "| service sống:", srv.poll() is None, flush=True)
    time.sleep(60)